# Day 5 — WaveNet (makemore Part 5, 56m)

Karpathy starts from the part-3 pytorchified code and grows it: context 3 -> 8, then flat -> hierarchical (WaveNet-style tree). New classes get built DURING the video — those are mine to write. Layers below are pasted at their part-3 state on purpose.

**targets (his performance log):** flat 8-context: train 1.918 / val 2.027 · hierarchical same params: val 2.029 · after his mid-video fix: val 2.022 · scaled (emb 24, hidden 128, 76K params): train 1.769 / **val 1.993**. beat 1.993.

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# ---- dashboard ----
BLOCK_SIZE = 8          # context length (3 -> 8, the whole point)
N_EMBD = 10
N_HIDDEN = 300
MAX_STEPS = 200000
BATCH_SIZE = 32
LEARNING_RATE = 0.1

In [4]:
# read words, build vocab
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(len(words), vocab_size)

32033 27


In [5]:
# shuffle + split at the top, like always
import random
random.seed(42)
random.shuffle(words)

def build_dataset(word_list, block_size):
  X, Y = [], []
  for w in word_list:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,  Ytr  = build_dataset(words[:n1], BLOCK_SIZE)
Xdev, Ydev = build_dataset(words[n1:n2], BLOCK_SIZE)
Xte,  Yte  = build_dataset(words[n2:], BLOCK_SIZE)

torch.Size([182625, 8]) torch.Size([182625])
torch.Size([22655, 8]) torch.Size([22655])
torch.Size([22866, 8]) torch.Size([22866])


In [6]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

........ --> y
.......y --> u
......yu --> h
.....yuh --> e
....yuhe --> n
...yuhen --> g
..yuheng --> .
........ --> d
.......d --> i
......di --> o
.....dio --> n
....dion --> d
...diond --> r
..diondr --> e
.diondre --> .
........ --> x
.......x --> a
......xa --> v
.....xav --> i
....xavi --> e


In [7]:
# my part-3 layers, pasted as-is (each line's ancestor lives in my bn.ipynb loop)
# -----------------------------------------------------------------------------
class Linear:

  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5 # kaiming
    self.bias = torch.zeros(fan_out) if bias else None

  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out

  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])

# -----------------------------------------------------------------------------
class BatchNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (running momentum update)
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)

  def __call__(self, x):
    if self.training:
      xmean = x.mean(0, keepdim=True)
      xvar = x.var(0, keepdim=True)
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

# -----------------------------------------------------------------------------
class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

In [8]:
# ---- my classes ----
# class Embedding: ...
# class FlattenConsecutive: ...
# class Sequential: ...

In [9]:
torch.manual_seed(42);

In [10]:
# ---- my model ----
# flat first, hierarchical after
# model = Sequential([...])

# param init
# with torch.no_grad():
#   model.layers[-1].weight *= 0.1  # last layer less confident

# parameters = model.parameters()
# print(sum(p.nelement() for p in parameters))
# for p in parameters:
#   p.requires_grad = True

In [11]:
# training loop (same as last time)
lossi = []
for i in range(MAX_STEPS):
  ix = torch.randint(0, Xtr.shape[0], (BATCH_SIZE,))
  Xb, Yb = Xtr[ix], Ytr[ix]

  logits = model(Xb)
  loss = F.cross_entropy(logits, Yb)

  for p in parameters:
    p.grad = None
  loss.backward()

  lr = LEARNING_RATE if i < 150000 else LEARNING_RATE / 10
  for p in parameters:
    p.data += -lr * p.grad

  if i % 10000 == 0:
    print(f'{i:7d}/{MAX_STEPS:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

NameError: name 'model' is not defined

In [ ]:
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))

In [ ]:
# eval mode, then split losses
for layer in model.layers:
  layer.training = False

@torch.no_grad()
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  logits = model(x)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

## observations

- (runs go here)

In [ ]:
# sample
g = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(20):
  out = []
  context = [0] * BLOCK_SIZE
  while True:
    logits = model(torch.tensor([context]))
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1, generator=g).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix == 0:
      break
  print(''.join(itos[i] for i in out))